# Setup R & Random Forest Dependencies

This notebook sets up the environment and installs all necessary system and R packages required to train Random Forest models on `.RData` datasets, both locally and in **Kaggle** / **Google Colab** environments.

### Dependencies Installed:
- **R Base & Dev Tools**: `r-base`, `r-base-dev`
- **Random Forest & ML Libraries**: `randomForest`, `ranger`, `caret`, `e1071`
- **Data Processing & Config Utilities**: `jsonlite`, `dplyr`, `data.table`
- **Visualization & Evaluation**: `ggplot2`, `pROC`
- **Jupyter R Kernel**: `IRkernel`

In [ ]:
%%bash
# Step 1: Add CRAN repository and install R base system packages
export DEBIAN_FRONTEND=noninteractive

echo "===> Updating APT repositories and installing build tools..."
sudo apt-get update -qq
sudo apt-get install -y --no-install-recommends \
    software-properties-common \
    dirmngr \
    wget \
    curl \
    ca-certificates \
    build-essential \
    libcurl4-openssl-dev \
    libssl-dev \
    libxml2-dev

echo "===> Adding CRAN repository key and repository..."
wget -qO- https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc | sudo tee /etc/apt/trusted.gpg.d/cran_ubuntu_key.asc > /dev/null
sudo add-apt-repository -y "deb https://cloud.r-project.org/bin/linux/ubuntu $(lsb_release -cs)-cran40/"
sudo apt-get update -qq

echo "===> Installing R base and development packages..."
sudo apt-get install -y --no-install-recommends r-base r-base-dev

In [ ]:
%%bash
# Step 2: Install fast prebuilt R binary packages via APT (recommended for Kaggle/Ubuntu)
export DEBIAN_FRONTEND=noninteractive

echo "===> Installing R prebuilt binary packages via APT..."
sudo apt-get install -y --no-install-recommends \
    r-cran-randomforest \
    r-cran-ranger \
    r-cran-caret \
    r-cran-e1071 \
    r-cran-jsonlite \
    r-cran-dplyr \
    r-cran-ggplot2 \
    r-cran-proc \
    r-cran-data.table \
    r-cran-irkernel || true

In [ ]:
%%bash
# Step 3: Verify and install missing packages via CRAN & register Jupyter IRkernel
Rscript -e '
required_pkgs <- c("randomForest", "ranger", "caret", "e1071", "jsonlite", "dplyr", "ggplot2", "pROC", "data.table", "IRkernel")
missing_pkgs <- required_pkgs[!(required_pkgs %in% installed.packages()[,"Package"])]

if (length(missing_pkgs) > 0) {
  cat("Installing missing R packages from CRAN:", paste(missing_pkgs, collapse=", "), "\n")
  install.packages(missing_pkgs, repos="https://cloud.r-project.org", Ncpus = parallel::detectCores())
} else {
  cat("All required R packages are already installed!\n")
}

# Register IRkernel for Jupyter / Kaggle
if (requireNamespace("IRkernel", quietly = TRUE)) {
  cat("Registering IRkernel for Jupyter notebook...\n")
  tryCatch({
    IRkernel::installspec(user = FALSE)
  }, error = function(e) {
    IRkernel::installspec(user = TRUE)
  })
  cat("IRkernel registered successfully!\n")
}
'

In [ ]:
%%bash
# Step 4: Verification check of R installation and required packages
echo "===> Verification Report:"
Rscript -e '
cat("R Version:", R.version.string, "\n\n")
cat(sprintf("%-18s %-12s\n", "Package", "Status"))
cat(paste(rep("-", 30), collapse=""), "\n")
pkgs <- c("randomForest", "ranger", "caret", "e1071", "jsonlite", "dplyr", "ggplot2", "pROC", "data.table", "IRkernel")
all_ok <- TRUE
for (p in pkgs) {
  avail <- requireNamespace(p, quietly = TRUE)
  if (!avail) all_ok <- FALSE
  cat(sprintf("%-18s %-12s\n", p, ifelse(avail, "[OK]", "[MISSING]")))
}
cat(paste(rep("-", 30), collapse=""), "\n")
if (all_ok) {
  cat("SUCCESS: All dependencies are ready for R Random Forest training in models/rf.ipynb!\n")
} else {
  cat("WARNING: Some packages are missing. Please re-run CRAN installation step.\n")
}
'